<a href="https://colab.research.google.com/github/MohammedMujtabaAnsari/MACHINE-LEARNING-/blob/main/smart_tech_project_Device_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Install/Upgrade transformers and torch to resolve potential compatibility issues.
# This specific RuntimeError often indicates a version conflict between torch and transformers.
# A runtime restart is usually required after this step for changes to take full effect.
!pip install --upgrade transformers torch

# Now proceed with the imports.
from transformers import AutoTokenizer, AutoModel
import torch
from datasets import load_dataset

In [ ]:

np.random.seed(42)
n_samples = 1000
data = {
    'power_usage': np.random.normal(50, 20, n_samples),
    'traffic_bytes': np.random.exponential(1000, n_samples),
    'response_time': np.random.uniform(10, 500, n_samples),
    'text_desc': ['voice-activated thermostat', 'security camera motion detect', 'smart light dimmer',
                  'door sensor alert', 'fridge temp monitor'] * (n_samples//5) + list(np.random.choice(['iot bulb', 'ac unit'], n_samples%5)),
    'device_type': np.random.choice(['thermostat', 'camera', 'light', 'sensor', 'fridge'], n_samples),  # Classification target
    'failure_risk': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])  # Prediction target (binary)
}
df = pd.DataFrame(data)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModel.from_pretrained('distilbert-base-uncased')

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=32)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()[:128]  # 128-dim embedding

df['nlp_emb'] = df['text_desc'].apply(get_embedding)
emb_df = pd.DataFrame(df['nlp_emb'].tolist())  # To columns


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

X_class = pd.concat([df[['power_usage', 'traffic_bytes', 'response_time']], emb_df], axis=1)
y_class = df['device_type'].astype('category').cat.codes
X_pred = X_class.copy()
y_pred = df['failure_risk']


In [ ]:
X_class.columns = X_class.columns.astype(str)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)
clf_class = RandomForestClassifier(n_estimators=100, random_state=42)
clf_class.fit(X_train_c, y_train_c)
pred_class = clf_class.predict(X_test_c)
print("Classification Accuracy:", accuracy_score(y_test_c, pred_class))
print(classification_report(y_test_c, pred_class))


Classification Accuracy: 0.22
              precision    recall  f1-score   support

           0       0.23      0.24      0.23        42
           1       0.17      0.19      0.18        42
           2       0.28      0.22      0.25        41
           3       0.22      0.25      0.24        40
           4       0.21      0.20      0.21        35

    accuracy                           0.22       200
   macro avg       0.22      0.22      0.22       200
weighted avg       0.22      0.22      0.22       200



In [ ]:
X_pred.columns = X_pred.columns.astype(str)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pred, y_pred, test_size=0.2, random_state=42)
clf_pred = RandomForestClassifier(n_estimators=100, random_state=42)
clf_pred.fit(X_train_p, y_train_p)
pred_risk = clf_pred.predict(X_test_p)
print("Prediction Accuracy:", accuracy_score(y_test_p, pred_risk))
print(classification_report(y_test_p, pred_risk))


Prediction Accuracy: 0.785
              precision    recall  f1-score   support

           0       0.81      0.96      0.88       163
           1       0.00      0.00      0.00        37

    accuracy                           0.79       200
   macro avg       0.40      0.48      0.44       200
weighted avg       0.66      0.79      0.72       200



In [ ]:
new_sample_data = np.array([[60, 1500, 200] + [0.1]*128])  # Numerical features + NLP embeddings

# Ensure new_sample has the same column names as the training data
# Use X_class.columns as it contains the names from both numerical features and NLP embeddings
new_sample_df = pd.DataFrame(new_sample_data, columns=X_class.columns)

device_pred = clf_class.predict(new_sample_df)[0]
risk_pred = clf_pred.predict(new_sample_df)[0]

device_names = ['thermostat', 'camera', 'light', 'sensor', 'fridge']
print(f"Predicted Device: {device_names[device_pred]}, Failure Risk: {'High' if risk_pred else 'Low'}")

Predicted Device: light, Failure Risk: Low
